In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import anndata as ad
import scirpy as ir
import tarfile
import warnings   
from glob import glob

import matplotlib.pyplot as plt
import muon as mu
import os

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ["NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

sc.settings.set_figure_params(dpi=100, facecolor="white")

## Load data

In [ ]:
path_root = "/path_to/scRNAseq"
file_out = f"{path_root}/03_TempOuts/ClusterV3_251015.h5mu"
mdata = mu.read(file_out)

sc.pl.umap(mdata["gex"], color=["cell_type_lvl3"])

## Cell Fraction

### All Cell

In [ ]:
SampleTable = mdata["gex"].obs[["sample", "cell_type_lvl3"]]
SampleTable["ClassCell"] = SampleTable["sample"].astype(str) + "_" + SampleTable["cell_type_lvl3"].astype(str)
SampleTable2 = pd.DataFrame(SampleTable["ClassCell"].value_counts())
SampleTable2["ClassCell"] = SampleTable2.index
SampleTable2 = SampleTable2.reset_index(drop=True)
SampleTable2[['sample', 'cell_type']] = SampleTable2['ClassCell'].str.split('_', expand=True)
SampleTable2 = SampleTable2[SampleTable2["cell_type"] != "Malignant"]

for SampleNM in pd.unique(SampleTable2["sample"]):

    SampleTable2.loc[SampleTable2["sample"] == SampleNM, "Prop"] = SampleTable2.loc[SampleTable2["sample"] == SampleNM, "count"] / \
                                                                    SampleTable2.loc[SampleTable2["sample"] == SampleNM, "count"].sum()

for CellType in pd.unique(SampleTable2["cell_type"]):
    for sample in ["Chd6KD", "Chd6Scr"]:
        SampleTable2.loc[SampleTable2["ClassCell"] == sample + "_" + CellType, "Prop_CT"] = SampleTable2.loc[SampleTable2["ClassCell"] == sample + "_" + CellType, "Prop"] / \
                                                                        SampleTable2.loc[SampleTable2["cell_type"] == CellType, "Prop"].sum()



current_categories = ["Endothelial", "Neutrophil", "T cell", "NK", "Mono_Macrophage", "DC", "pDC", "B cell"]

SampleTable2['cell_type'] = SampleTable2['cell_type'].astype("category")
SampleTable2['cell_type'] = SampleTable2['cell_type'].cat.set_categories(current_categories)

SampleTable2 = SampleTable2.pivot_table(index = "sample", columns="cell_type", values="Prop_CT").T
SampleTable2 = SampleTable2.reset_index()
SampleTable2.columns.name = None
SampleTable2 = SampleTable2[["cell_type", "Chd6Scr", "Chd6KD"]]


sns.set(style='white')
plt.figure(figsize=(6, 6))
#create stacked bar chart
SampleTable2.set_index('cell_type').plot(kind='barh', stacked=True)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1)

path_root = "/path_to/scRNAseq"
path_plot = f"{path_root}/06_plot/CellProp/CellProportion_all.svg"
plt.savefig(path_plot, bbox_inches='tight')

## T cell subset

In [ ]:
SampleTable = mdata["gex"].obs[["sample", "cell_type_lvl2"]]
SampleTable["ClassCell"] = SampleTable["sample"].astype(str) + "_" + SampleTable["cell_type_lvl2"].astype(str)
SampleTable2 = pd.DataFrame(SampleTable["ClassCell"].value_counts())
SampleTable2["ClassCell"] = SampleTable2.index
SampleTable2 = SampleTable2.reset_index(drop=True)

SampleTable2[['sample', 'cell_type']] = SampleTable2['ClassCell'].str.split('_', expand=True)

for SampleNM in pd.unique(SampleTable2["sample"]):

    SampleTable2.loc[SampleTable2["sample"] == SampleNM, "Prop"] = SampleTable2.loc[SampleTable2["sample"] == SampleNM, "count"] / \
                                                                    SampleTable2.loc[SampleTable2["sample"] == SampleNM, "count"].sum()


file_out = f"{path_root}/03_TempOuts/plot_out/CellProp_plot/Tsubset_ImmuneCellALL.tsv"
SampleTable2.to_csv(file_out, sep = "\t", index = False)

SampleTable2["sample"] = SampleTable2["sample"].astype("category")
SampleTable2["sample"] = SampleTable2["sample"].cat.set_categories(["Chd6Scr", "Chd6KD"])

SampleTable2 = SampleTable2[SampleTable2["cell_type"].isin([
                                    "Cd4+ Naive T", "Cd4+ Treg", "Cd4+ Treg Proliferating", "Nkg7+ Th1", "Nkg7+ Th1 Proliferating", 
                                    "Cd8+ Naive T", "Cd8+ Cytoxic T", "Cd8+ Exhausted T", 
                                    "NKT", 
                                    "Gamma Delta T", "Cytoxic T", "Cxcl10+ T", "Effector Memory T", "Cd74+ Memory T", 
                                    ])] 


plt.figure(figsize=(8, 6))
ax = sns.barplot(x="cell_type", y="Prop", hue="sample", data=SampleTable2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center')
ax.legend(loc='upper right')
ax.set(xlabel='Cell type', ylabel='Proportion')

plot_out = f"{path_root}/03_TempOuts/plot_out/CellProp_plot/Tsubset_ImmuneCellALL.svg"
plt.savefig(plot_out, bbox_inches='tight')

## Malignant DEG

In [ ]:
adata_sub = mdata[mdata.obs["gex:cell_type_lvl3"].isin(["Malignant"])]
sc.pl.umap(adata_sub["gex"], color=["cell_type_lvl3"])
adata_sub.update()

In [ ]:
adata_sub["gex"].X = adata_sub["gex"].layers["counts"].copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata_sub["gex"], target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata_sub["gex"])

adata_sub.raw = adata_sub.copy()

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata_sub["gex"], groupby="sample", method="wilcoxon", pts=True)

DEG_df = sc.get.rank_genes_groups_df(adata_sub["gex"], group="Chd6KD")

file_out = f"{path_root}/03_TempOuts/DEGene_res/01_preData/DEGenes_Malignant_DiSample.tsv"
DEG_df.to_csv(file_out, sep = "\t", index=False)

## Malignant Hypoxia score

In [ ]:
GeneList_Hypoxia = ["Ackr3", "Adm", "Adora2b", "Ak4", "Akap12", "Aldoa", "Aldob", 
                    "Aldoc", "Ampd3", "Angptl4", "Ankzf1", "Anxa2", "Atf3", "Atp7a", "B3galt6", "B4galnt2", 
                    "Bcan", "Bcl2", "Bgn", "Bhlhe40", "Bnip3l", "Brs3", "Btg1", "Car12", "Casp6", "Cav1", 
                    "Cavin1", "Cavin3", "Ccn1", "Ccn2", "Ccn5", "Ccng2", "Cdkn1a", "Cdkn1b", "Cdkn1c", 
                    "Chst2", "Chst3", "Cited2", "Col5a1", "Cp", "Csrp2", "Cxcr4", "Dcn", "Ddit3", "Ddit4", 
                    "Dpysl4", "Dtna", "Dusp1", "Edn2", "Efna1", "Efna3", "Egfr", "Eno1", "Eno2", "Eno3", 
                    "Ero1a", "Errfi1", "Ets1", "Ext1", "F3", "Fam162a", "Fbp1", "Fos", "Fosl2", "Foxo3", 
                    "Gaa", "Galk1", "Gapdh", "Gapdhs", "Gbe1", "Gck", "Gcnt2", "Glrx", "Gpc1", "Gpc3", 
                    "Gpc4", "Gpi1", "Grhpr", "Gys1", "Has1", "Hdlbp", "Hexa", "Hk1", "Hk2", "Hmox1", 
                    "Hoxb9", "Hs3st1", "Hspa5", "Ids", "Ier3", "Igfbp1", "Igfbp3", "Il6", "Ilvbl", 
                    "Inha", "Irs2", "Isg20", "Jmjd6", "Jun", "Kdelr3", "Kdm3a", "Kif5a", "Klf6", 
                    "Klf7", "Klhl24", "Lalba", "Large1", "Ldha", "Ldhc", "Lox", "Lxn", "Maff", 
                    "Map3k1", "Mif", "Mt1", "Mt2", "Mxi1", "Myh9", "Nagk", "Ncan", "Ndrg1", 
                    "Ndst1", "Ndst2", "Nedd4l", "Nfil3", "Noct", "Nr3c1", "P4ha1", "P4ha2", 
                    "Pam", "Pck1", "Pdgfb", "Pdk1", "Pdk3", "Pfkfb3", "Pfkl", "Pfkp", "Pgam2", 
                    "Pgf", "Pgk1", "Pgm1", "Pgm2", "Phkg1", "Pim1", "Pklr", "Pkp1", "Plac8", 
                    "Plaur", "Plin2", "Pnrc1", "Ppargc1a", "Ppfia4", "Ppp1r15a", "Ppp1r3c", "Prdx5", 
                    "Prkca", "Pygm", "Rbpj", "Rora", "Rragd", "S100a4", "Sap30", "Scarb1", "Sdc2", 
                    "Sdc3", "Sdc4", "Selenbp2", "Serpine1", "Siah2", "Slc25a1", "Slc2a1", "Slc2a3", 
                    "Slc2a5", "Slc37a4", "Slc6a6", "Srpx", "Stbd1", "Stc1", "Stc2", "Sult2b1", "Tes", 
                    "Tgfb3", "Tgfbi", "Tgm2", "Tiparp", "Tktl1", "Tmem45a", "Tnfaip3", "Tpbg", "Tpd52", 
                    "Tpi1", "Tpst2", "Ugp2", "Vegfa", "Vhl", "Vldlr", "Wsb1", "Xpnpep1", "Zfp36", "Zfp292"]


mdata_temp = mdata[mdata.obs['gex:cell_type_lvl3'] == "Malignant", :].copy()
mdata_temp["gex"].X = mdata_temp["gex"].layers["counts"].copy()
# Normalizing to median total counts
sc.pp.normalize_total(mdata_temp["gex"])
# Logarithmize the data
sc.pp.log1p(mdata_temp["gex"])
mdata_temp["gex"].raw = mdata_temp["gex"].copy()

sc.tl.score_genes(mdata_temp["gex"], GeneList_Hypoxia, score_name="HALLMARK_Hypoxia")
sc.pl.dotplot(mdata_temp["gex"], ["HALLMARK_Hypoxia"], groupby = ["cell_type_lvl3", "sample"], dendrogram=True, show=False)

## Endothelial Signature score

In [ ]:
adata_sub = mdata[mdata.obs["gex:cell_type_lvl3"].isin(["Endothelial"])]
sc.pl.umap(adata_sub["gex"], color=["cell_type_lvl3"])
adata_sub.update()

adata_sub["gex"].X = adata_sub["gex"].layers["counts"].copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata_sub["gex"], target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata_sub["gex"])

adata_sub.raw = adata_sub.copy()

In [ ]:
path_root = "/path_to/PubscRNA/01_Mouse.HCC"
file_out = f"{path_root}/05_DEG/Endo.Public_DEG.tsv"
DEG_df = pd.read_csv(file_out, sep = "\t")

Gene_Tumor = DEG_df.loc[(DEG_df["logfoldchanges"] > 1) & (DEG_df["pvals_adj"] < 0.05) & (DEG_df["pct_nz_group"] > 0.25) & (DEG_df["pct_nz_reference"] > 0.25), :]
Gene_Tumor = Gene_Tumor.head(50)
Gene_Tumor = Gene_Tumor["names"].to_list()
Gene_Normal = DEG_df.loc[(DEG_df["logfoldchanges"] < -1) & (DEG_df["pvals_adj"] < 0.05) & (DEG_df["pct_nz_group"] > 0.25) & (DEG_df["pct_nz_reference"] > 0.25), :]
Gene_Normal = Gene_Normal["names"].to_list()

Profileration_GeneSet = ["Mki67", "Top2a", "Cdkn2a", "Cdkn1b", "Cdk2", "Ccnb1", "Ccne1", "E2f1"]
VascularNormal_GeneSet = ["Cdh5", "Tek", "Epas1", "Kdr", "Flt4", "Ecscr", "Aoc3", "Cyyr1", "Foxo1"]

sc.tl.score_genes(adata_sub["gex"], VascularNormal_GeneSet, score_name='Vascular_Normalization_Score')
sc.tl.score_genes(adata_sub["gex"], Profileration_GeneSet, score_name='Proliferation_Score')
sc.tl.score_genes(adata_sub["gex"], Gene_Normal, score_name='Gene_Normal')
sc.tl.score_genes(adata_sub["gex"], Gene_Tumor, score_name='Gene_Tumor')

sc.pl.dotplot(adata_sub["gex"], ["Vascular_Normalization_Score", "Proliferation_Score", "Gene_Normal", "Gene_Tumor"], groupby = "sample", show = False, standard_scale="var") #, vmax = 0.1

path_root = "/path_to/scRNAseq"
path_plot = f"{path_root}/06_plot/Endothelial_SigScore/Endothelila_SignatureScoreDotplot_Std.svg"
plt.savefig(path_plot, bbox_inches='tight')

## T cell Clone Expansion

In [ ]:
path_root = "/path_to/scRNAseq"
file_out = f"{path_root}/03_TempOuts/adataT_Cluster.h5mu"
adata_T = mu.read(file_out)
adata_T.update()

In [ ]:
ir.pp.index_chains(adata_T)
ir.tl.chain_qc(adata_T)


adata_T.obs["airr:chain_pairing2"] = adata_T.obs["airr:chain_pairing"]
current_categories = ['orphan VJ', 'extra VJ', 'single pair', 'orphan VDJ', 'extra VDJ', 'two full chains', "No TCR"]
adata_T.obs["airr:chain_pairing2"] = adata_T.obs["airr:chain_pairing2"].astype("category")
adata_T.obs["airr:chain_pairing2"] = adata_T.obs["airr:chain_pairing2"].cat.set_categories(current_categories)
adata_T.obs.loc[adata_T.obs["airr:chain_pairing2"].isna(), "airr:chain_pairing2"] = "No TCR"
adata_T.obs["gex:sample"] = adata_T.obs["gex:sample"].cat.set_categories([ "Chd6KD", "Chd6Scr"], ordered=True)
adata_T.update()

In [ ]:
plota = ir.pl.group_abundance(adata_T, groupby="gex:sample", target_col="airr:chain_pairing", normalize = True)
plota.set(xlabel='Sample', ylabel='Proportion within TCR cell', title="TCR chains proportion")

path_root = "/path_to/scRNAseq"
path_plot = f"{path_root}/06_plot/TCR_plot/TCR_ChainProp_barplot.svg"
plt.savefig(path_plot, bbox_inches='tight')

In [ ]:
_ = ir.pl.group_abundance(adata_T, groupby="airr:chain_pairing", target_col="gex:sample")

mu.pp.filter_obs(adata_T, "airr:chain_pairing", lambda x: ~np.isin(x, ["orphan VJ"]))
# adata_T
_ = ir.pl.group_abundance(adata_T, groupby="airr:chain_pairing", target_col="gex:sample")

# using default parameters, `ir_dist` will compute nucleotide sequence identity
ir.pp.ir_dist(adata_T)
ir.tl.define_clonotypes(adata_T, receptor_arms="VDJ", dual_ir="primary_only")

In [ ]:
ir.tl.clonotype_network(adata_T, min_cells=2)
_ = ir.pl.clonotype_network(adata_T, color="gex:sample", base_size=20, label_fontsize=9, panel_size=(7, 7))

In [ ]:
ir.pp.ir_dist(
    adata_T,
    metric="tcrdist",
    sequence="aa",
    cutoff=15,
)

ir.tl.define_clonotype_clusters(adata_T, sequence="aa", metric="tcrdist", receptor_arms="VDJ", dual_ir="any")
ir.tl.clonotype_network(adata_T, min_cells=3, sequence="aa", metric="tcrdist")
_ = ir.pl.clonotype_network(adata_T, color="gex:sample", label_fontsize=9, panel_size=(7, 7), base_size=20)

In [ ]:
ir.tl.clonal_expansion(adata_T, breakpoints = (1, 10))
adata_T["airr"].obs["clonal_expansion2"] = adata_T["airr"].obs["clonal_expansion"]
adata_T["airr"].obs.loc[adata_T["airr"].obs["clonal_expansion"] == "<= 1", "clonal_expansion2"] = "<= 10"
adata_T.update()

adata_T.obs["airr:clonal_expansion2"] = adata_T.obs["airr:clonal_expansion2"].cat.set_categories(["<= 10", "> 10"])
adata_T.update()

In [ ]:
## split data
adataT_Scr = adata_T[adata_T.obs['gex:sample'] == "Chd6Scr", :]
adataT_KD = adata_T[adata_T.obs['gex:sample'] == "Chd6KD", :]
adataT_Scr.update()
adataT_KD.update()


## Chd6Scr
ir.pl.clonal_expansion(adataT_Scr, target_col="clone_id", groupby="gex:cell_type_T", breakpoints=(0, 10))

path_root = "/path_to/scRNAseq"
path_plot = f"{path_root}/06_plot/TCR_plot/TCR_CloneExpansionSizeNorm_barplot_Chd6Scr.svg"
plt.savefig(path_plot, bbox_inches='tight')

## Chd6KD
ir.pl.clonal_expansion(adataT_KD, target_col="clone_id", groupby="gex:cell_type_T", breakpoints=(0, 10))

path_plot = f"{path_root}/06_plot/TCR_plot/TCR_CloneExpansionSizeNorm_barplot_Chd6KD.svg"
plt.savefig(path_plot, bbox_inches='tight')